# GARCH & Merton vs Reference Comparison

Compares generated financial time-series from two classical models against the ground-truth reference:
- **Reference** — real log-returns from `SNP500_individual_normalized_replication` (one CSV per ticker, `close` column, z-scored → unnormalized)
- **GARCH** — GARCH-X samples (raw log-returns)
- **Merton** — Merton jump-diffusion samples (raw log-returns)

**Part 1** — Aggregate statistics of generated values and increments.
**Part 2** — Marginal distribution comparison (KS, Wasserstein-1) vs Reference.
**Part 3** — Stylized facts overlap (fat tails, volatility clustering, leverage effect).

In [1]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────

# Directory with one CSV per ticker; each has a 'close' column of z-scored log-returns
REFERENCE_DIRECTORY = '../Master-Thesis/data/SNP500_individual_normalized_replication'

# Normalization stats used to unnormalize the reference z-scores
# CSV with index = feature name, columns = [mean, std]
UNNORMALIZATION_STATS_PATH = (
    '../Master-Thesis/data/general/normalization_stats_SNP500_super_normalized.csv'
)

# Classical model generated CSV paths
GARCH_DIRECTORY  = '../Master-Thesis/data/generated/garch_replication/generated_close.csv'
MERTON_DIRECTORY = '../Master-Thesis/data/generated/merton_replication/generated_close.csv'

# Parquet cache paths (set extract_df = True on first run to build them)
extract_df     = False
GARCH_PARQUET  = '../Master-Thesis/data/generated/garch_replication/generated_close.parquet'
MERTON_PARQUET = '../Master-Thesis/data/generated/merton_replication/generated_close.parquet'

# Output directories (created automatically)
appendix         = 'garch_merton_replication'
OUTPUT_IMAGE_DIR = '../Master-Thesis/images/generated/garch_merton_comparison'
OUTPUT_TABLE_DIR = '../Master-Thesis/tables/generated/garch_merton_comparison'
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from IPython.display import display

sys.path.insert(0, str(Path('..').resolve()))
import replication.stylized_facts as sf

warnings.filterwarnings('ignore', category=RuntimeWarning)

Path(OUTPUT_IMAGE_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_TABLE_DIR).mkdir(parents=True, exist_ok=True)

MODEL_ORDER  = ['Reference', 'GARCH', 'Merton']
MODEL_COLORS = {'Reference': 'steelblue', 'GARCH': 'forestgreen', 'Merton': 'crimson'}

In [3]:
# ── Utility functions ─────────────────────────────────────────────────────────

def compute_moments(x, label):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {'model': label}
    return {
        'model'           : label,
        'n'               : len(x),
        'mean'            : np.mean(x),
        'std'             : np.std(x),
        'variance'        : np.var(x),
        'skewness'        : float(scipy_stats.skew(x)),
        'excess_kurtosis' : float(scipy_stats.kurtosis(x, fisher=True)),
        'IQR'             : float(np.quantile(x, 0.75) - np.quantile(x, 0.25)),
        'min'             : np.min(x),
        'q001'            : np.quantile(x, 0.001),
        'q01'             : np.quantile(x, 0.01),
        'q05'             : np.quantile(x, 0.05),
        'q25'             : np.quantile(x, 0.25),
        'q50'             : np.quantile(x, 0.50),
        'q75'             : np.quantile(x, 0.75),
        'q95'             : np.quantile(x, 0.95),
        'q99'             : np.quantile(x, 0.99),
        'q999'            : np.quantile(x, 0.999),
        'max'             : np.max(x),
    }


def subsample(arr, max_points=50_000_000, seed=42):
    arr = arr[np.isfinite(arr)]
    if len(arr) > max_points:
        arr = np.random.default_rng(seed).choice(arr, size=max_points, replace=False)
    return arr


def to_flat(df, step_cols, max_points=50_000_000, seed=42):
    vals = df[step_cols].values.ravel().astype(float)
    return subsample(vals, max_points, seed)


def to_increments(df, step_cols, max_points=1_000_000, seed=42):
    arr  = df[step_cols].values.astype(float)
    incs = np.diff(arr, axis=1).ravel()
    return subsample(incs, max_points, seed)


def to_paths_obj(df, step_cols, max_paths=10000, seed=42):
    arr = df[step_cols].values.astype(float)
    n   = len(arr)
    if n > max_paths:
        idx = np.random.default_rng(seed).choice(n, size=max_paths, replace=False)
        arr = arr[idx]
    paths = np.empty(len(arr), dtype=object)
    for i, row in enumerate(arr):
        paths[i] = row[np.isfinite(row)]
    return paths


def save_fig(fig, name):
    path = Path(OUTPUT_IMAGE_DIR) / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')

## 1. Load Data

### 1a. Reference

Each CSV in `REFERENCE_DIRECTORY` contains one ticker's full history. The `close` column holds
z-scored log-returns; they are unnormalized via `x = x_norm * std_close + mean_close` before comparison.

In [4]:
# ── Normalization statistics ───────────────────────────────────────────────────
stats_df = pd.read_csv(UNNORMALIZATION_STATS_PATH, index_col=0)
stats_df.columns = ['mean', 'std']
stats_df.index   = stats_df.index.str.strip().str.lower()
NORM_STATS = stats_df.to_dict('index')
mu_close   = NORM_STATS['close']['mean']
sig_close  = NORM_STATS['close']['std']
print(f'Close unnorm: x = x_norm * {sig_close:.6f} + {mu_close:.6f}')
display(stats_df)

Close unnorm: x = x_norm * 0.023162 + 0.000482


,mean,std
close,0.000482,0.023162
open,0.000303,0.012624
high,0.013087,0.020523
low,-0.012493,0.020123


In [5]:
# ── Load reference CSVs ───────────────────────────────────────────────────────
ref_csv_files = sorted(Path(REFERENCE_DIRECTORY).glob('*.csv'))
print(f'Found {len(ref_csv_files)} ticker files in reference directory')

ref_series = []   # list of 1-D arrays, one per ticker (unnormalized log-returns)
for csv_file in ref_csv_files:
    df = pd.read_csv(csv_file)
    vals = df['close'].values.astype(float)
    vals = vals[np.isfinite(vals)]
    vals = vals * sig_close + mu_close   # unnormalize
    ref_series.append(vals)

# flat reference data
ref_flat_full = np.concatenate(ref_series)
ref_flat_full = ref_flat_full[np.isfinite(ref_flat_full)]

# increments per ticker (first differences), then concatenated
ref_inc_full = np.concatenate([np.diff(s) for s in ref_series if len(s) > 1])
ref_inc_full = ref_inc_full[np.isfinite(ref_inc_full)]

# paths for stylized facts (one path = one ticker's full series)
ref_paths_all = np.empty(len(ref_series), dtype=object)
for i, s in enumerate(ref_series):
    ref_paths_all[i] = s

v = ref_flat_full
print(f'\n[Reference] tickers={len(ref_series)}  '
      f'total_points={len(v):,}')
print(f'  scale check: mean={v.mean():.6f}  std={v.std():.6f}  '
      f'range=[{v.min():.4f}, {v.max():.4f}]')

Found 210 ticker files in reference directory

[Reference] tickers=210  total_points=2,623,569
  scale check: mean=0.000482  std=0.023162  range=[-1.0393, 0.7694]


### 1b. GARCH and Merton

In [6]:
def _load_csv_and_save(csv_path: str, parquet_path: str) -> pd.DataFrame:
    print(f'[csv] Reading {Path(csv_path).name} (slow) …')
    df = pd.read_csv(csv_path, low_memory=False)
    step_cols = [c for c in df.columns if c.startswith('step_')]
    df = df[df['step_000'] != 'step_000'].reset_index(drop=True)
    df[step_cols] = df[step_cols].astype('float32')
    df['ticker'] = df['file'].str.split('_').str[0]
    df.to_parquet(parquet_path, index=False)
    print(f'[csv] Saved cache → {parquet_path}')
    return df


if extract_df:
    df_garch  = _load_csv_and_save(GARCH_DIRECTORY,  GARCH_PARQUET)
    df_merton = _load_csv_and_save(MERTON_DIRECTORY, MERTON_PARQUET)
else:
    print(f'[parquet] Loading {GARCH_PARQUET}')
    df_garch  = pd.read_parquet(GARCH_PARQUET)
    print(f'[parquet] Loading {MERTON_PARQUET}')
    df_merton = pd.read_parquet(MERTON_PARQUET)

GARCH_STEP_COLS  = [c for c in df_garch.columns  if c.startswith('step_')]
MERTON_STEP_COLS = [c for c in df_merton.columns if c.startswith('step_')]
SEQ_LEN = 512

print(f'[GARCH]  rows={len(df_garch):,}  seq_len={len(GARCH_STEP_COLS)}')
print(f'[Merton] rows={len(df_merton):,}  seq_len={len(MERTON_STEP_COLS)}')

[parquet] Loading ../Master-Thesis/data/generated/garch_replication/generated_close.parquet
[parquet] Loading ../Master-Thesis/data/generated/merton_replication/generated_close.parquet
[GARCH]  rows=252,260  seq_len=512
[Merton] rows=252,260  seq_len=512


In [7]:
# Scale diagnostics — expected std ~ 0.023 for log-returns
print('=' * 72)
print('  SCALE DIAGNOSTICS  (expected std ~ 0.023 for log-returns)')
print('=' * 72)
diag_data = [
    ('Reference', ref_flat_full),
    ('GARCH',     df_garch[GARCH_STEP_COLS].values.ravel().astype(float)),
    ('Merton',    df_merton[MERTON_STEP_COLS].values.ravel().astype(float)),
]
for label, flat in diag_data:
    flat = flat[np.isfinite(flat)]
    pct_extreme = (np.abs(flat) > 1.0).mean() * 100
    flag = '  *** SCALE WARNING ***' if np.std(flat) > 0.5 else ''
    print(f'  {label:10s}: n={len(flat):>10,}  '
          f'mean={np.mean(flat):>9.4f}  std={np.std(flat):>8.4f}  '
          f'range=[{np.min(flat):>8.4f}, {np.max(flat):>8.4f}]  '
          f'|x|>1: {pct_extreme:.2f}%{flag}')
print()

  SCALE DIAGNOSTICS  (expected std ~ 0.023 for log-returns)
  Reference : n= 2,623,569  mean=   0.0005  std=  0.0232  range=[ -1.0393,   0.7694]  |x|>1: 0.00%
  GARCH     : n=129,157,120  mean=   0.0009  std=  0.0231  range=[-17.5946,  20.0961]  |x|>1: 0.00%
  Merton    : n=129,157,120  mean=   0.0005  std=  0.0366  range=[ -8.3654,   8.9261]  |x|>1: 0.02%



---
## Part 1 — Aggregate Statistics

In [8]:
flat_data = {
    'Reference': subsample(ref_flat_full),
    'GARCH'    : to_flat(df_garch,  GARCH_STEP_COLS),
    'Merton'   : to_flat(df_merton, MERTON_STEP_COLS),
}

df_agg = pd.DataFrame(
    [compute_moments(flat_data[m], m) for m in MODEL_ORDER]
).set_index('model')

print('Aggregate statistics on generated values:')
display(df_agg.T.round(6))

agg_path = Path(OUTPUT_TABLE_DIR) / f'aggregate_statistics_{appendix}.csv'
df_agg.to_csv(agg_path)
print(f'\nSaved: {agg_path}')

Aggregate statistics on generated values:


model,Reference,GARCH,Merton
n,2.623569e+06,5.000000e+07,5.000000e+07
mean,4.820000e-04,9.420000e-04,4.620000e-04
std,2.316200e-02,2.299000e-02,3.644900e-02
variance,5.360000e-04,5.290000e-04,1.329000e-03
skewness,-5.772670e-01,-7.563495e+00,1.210231e+00
excess_kurtosis,3.850281e+01,1.102170e+04,3.736487e+03
IQR,2.010800e-02,1.888900e-02,1.870300e-02
min,-1.039285e+00,-1.759463e+01,-8.276762e+00
q001,-1.371890e-01,-1.206640e-01,-1.521510e-01
q01,-6.262400e-02,-5.766400e-02,-5.844000e-02



Saved: ..\Master-Thesis\tables\generated\garch_merton_comparison\aggregate_statistics_garch_merton_replication.csv


### Aggregate statistics on increments

Increments: `x_t - x_{t-1}` along each path (for GARCH/Merton) or each ticker series (for Reference).

In [9]:
inc_data = {
    'Reference': subsample(ref_inc_full),
    'GARCH'    : to_increments(df_garch,  GARCH_STEP_COLS),
    'Merton'   : to_increments(df_merton, MERTON_STEP_COLS),
}

df_inc = pd.DataFrame(
    [compute_moments(inc_data[m], m) for m in MODEL_ORDER]
).set_index('model')

print('Aggregate statistics on increments (x_t - x_{t-1}):')
display(df_inc.T.round(6))

inc_path = Path(OUTPUT_TABLE_DIR) / f'increment_statistics_{appendix}.csv'
df_inc.to_csv(inc_path)
print(f'Saved: {inc_path}')

Aggregate statistics on increments (x_t - x_{t-1}):


model,Reference,GARCH,Merton
n,2.623359e+06,1000000.000000,1000000.000000
mean,-1.000000e-06,0.000011,0.000043
std,3.292200e-02,0.032437,0.050706
variance,1.084000e-03,0.001052,0.002571
skewness,2.227170e-01,-1.751990,2.098479
excess_kurtosis,2.941000e+01,1500.627045,1850.848739
IQR,2.949900e-02,0.028328,0.028991
min,-1.538815e+00,-5.521814,-5.783219
q001,-1.844710e-01,-0.162163,-0.280919
q01,-8.936200e-02,-0.081968,-0.079970


Saved: ..\Master-Thesis\tables\generated\garch_merton_comparison\increment_statistics_garch_merton_replication.csv


---
## Part 2 — Marginal Distribution Comparison

- **KS statistic**: maximum absolute difference between empirical CDFs (smaller = more similar).
- **Wasserstein-1**: expected absolute cost to transport one distribution to the other.

Both metrics are computed for GARCH and Merton against the Reference.

In [15]:
KS_MAX_POINTS = 1_000_000
ref_flat_ks = subsample(flat_data['Reference'], KS_MAX_POINTS)

ks_rows = []
for model in ['GARCH', 'Merton']:
    model_ks = subsample(flat_data[model], KS_MAX_POINTS)
    ks_stat, ks_p = scipy_stats.ks_2samp(model_ks, ref_flat_ks)
    w1 = scipy_stats.wasserstein_distance(model_ks, ref_flat_ks)
    ks_rows.append({'model'        : model,
                    'KS_stat'      : round(ks_stat, 4),
                    'KS_pvalue'    : f'{ks_p:.2e}',
                    'Wasserstein_1': round(w1, 6)})

df_ks = pd.DataFrame(ks_rows).set_index('model')
print(f'Distributional distances vs Reference (subsample n={KS_MAX_POINTS:,}):')
display(df_ks)
df_ks.to_csv(Path(OUTPUT_TABLE_DIR) / f'distributional_distances_{appendix}.csv')


Distributional distances vs Reference (subsample n=1,000,000):


,KS_stat,KS_pvalue,Wasserstein_1
model,,,
GARCH,0.0540,0.00e+00,0.001265
Merton,0.0423,0.00e+00,0.001881


---
## Part 3 — Stylized Facts Overlap

Computed via `replication.stylized_facts`:

- **`sf.distribution`**: normalized PDF of return values — reveals fat tails.
- **`sf.acf`**: mean ACF of `|r_t|` across paths — reveals volatility clustering.
- **`sf.leverage_effect`**: `L(t) = Corr(r_s, |r_{s+t}|^2)` — negative for real equities.

Reference paths = one path per ticker (full history). GARCH/Merton paths = one path per generated window.

In [26]:
MAX_SF_PATHS = 20000
MAX_LAG_ACF  = 1000
MAX_LAG_LEV  = 100

# Full data — used for the marginal distribution (fat-tail) plot
sf_flat = {
    'Reference': flat_data['Reference'],
    'GARCH'    : flat_data['GARCH'],
    'Merton'   : flat_data['Merton'],
}

# Subsampled paths — used for ACF and leverage (path-based computations)
sf_paths = {
    'Reference': ref_paths_all,
    'GARCH'    : to_paths_obj(df_garch,  GARCH_STEP_COLS,  max_paths=MAX_SF_PATHS),
    'Merton'   : to_paths_obj(df_merton, MERTON_STEP_COLS, max_paths=MAX_SF_PATHS),
}

print('Path counts for stylized facts:')
for m, arr in sf_paths.items():
    lens = [len(p) for p in arr]
    print(f'  {m}: {len(arr)} paths, len [{min(lens)}, {max(lens)}]')


Path counts for stylized facts:
  Reference: 210 paths, len [10083, 16175]
  GARCH: 20000 paths, len [512, 512]
  Merton: 20000 paths, len [512, 512]


In [27]:
# sf.distribution — fat-tail PDF
dist_results = {}
for model in MODEL_ORDER:
    out = str(Path(OUTPUT_IMAGE_DIR) / f'sf_distribution_{model}_{appendix}')
    dx, dy = sf.distribution(
        sf_flat[model], file_name=out,
        scale='log', multiple=False, normalize=True, granuality=100,
    )
    dist_results[model] = (dx, dy)
    print(f'  {model}: sf.distribution saved')

  Reference: sf.distribution saved
  GARCH: sf.distribution saved
  Merton: sf.distribution saved


In [28]:
# sf.acf — volatility clustering (ACF of |r_t|)
acf_results = {}
for model in MODEL_ORDER:
    out = str(Path(OUTPUT_IMAGE_DIR) / f'sf_acf_{model}_{appendix}')
    res = sf.acf(
        sf_paths[model], file_name=out,
        for_abs=True, multiple=True, fit=False,
        scale='log', max_lag=MAX_LAG_ACF,#min(MAX_LAG_ACF, SEQ_LEN // 2),
    )
    acf_results[model] = res
    print(f'  {model}: sf.acf saved  (shape={res.shape})')

  Reference: sf.acf saved  (shape=(1000,))
  GARCH: sf.acf saved  (shape=(1000,))
  Merton: sf.acf saved  (shape=(1000,))


In [29]:
# sf.leverage_effect — L(t) should be negative for equities
lev_results = {}
for model in MODEL_ORDER:
    out = str(Path(OUTPUT_IMAGE_DIR) / f'sf_leverage_{model}_{appendix}')
    res = sf.leverage_effect(
        sf_paths[model], file_name=out,
        multiple=True, min_lag=1, max_lag=MAX_LAG_LEV,
    )
    lev_results[model] = res
    print(f'  {model}: sf.leverage_effect saved  (shape={res.shape})')

  Reference: sf.leverage_effect saved  (shape=(99,))
  GARCH: sf.leverage_effect saved  (shape=(99,))
  Merton: sf.leverage_effect saved  (shape=(99,))


In [35]:
# Overlap comparison — all three on the same axes
lag_axis = np.arange(1, MAX_LAG_ACF + 1)
lev_lags = np.arange(1, MAX_LAG_LEV)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fat-tail distribution (positive side, log-log)
ax = axes[0]
for model in MODEL_ORDER:
    dx, dy = dist_results[model]
    mask = dx > 0
    ax.plot(dx[mask], dy[mask], '.', ms=3,
            color=MODEL_COLORS[model], alpha=0.8, label=model)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Normalised scale (sigma)')
ax.set_ylabel('PDF P(r)')
ax.set_title('Fat-tail distribution (log-log, positive side)')
ax.legend()


# ACF of |r_t|
ax = axes[1]
for model in MODEL_ORDER:
    acf_vals = np.abs(acf_results[model])
    ax.plot(lag_axis, acf_vals, '.', ms=3,
            color=MODEL_COLORS[model], alpha=0.8, label=model)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Lag k')
ax.set_ylabel('ACF of |r_t|')
ax.set_title('Volatility clustering — ACF overlap')
ax.legend()

# Leverage effect
ax = axes[2]
for model in MODEL_ORDER:
    ax.plot(lev_lags, lev_results[model],
            color=MODEL_COLORS[model], lw=1.4, label=model)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Lag t')
ax.set_ylabel('L(t)')
ax.set_title('Leverage effect — all models')
ax.legend()

plt.suptitle('Stylized facts comparison — Reference, GARCH, Merton', y=1.02)
plt.tight_layout()
save_fig(fig, f'stylized_facts_overlap_{appendix}')
display(fig); plt.close(fig)

Saved: ..\Master-Thesis\images\generated\garch_merton_comparison\stylized_facts_overlap_garch_merton_replication.png


<Figure size 1800x500 with 3 Axes>

# Marginal Distribution

In [32]:
lo, hi = np.quantile(flat_data['Reference'], [0.001, 0.999])
bins   = np.linspace(lo, hi, 80)
probs  = np.linspace(0.001, 0.999, 2_000)

for model in ['GARCH', 'Merton']:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    color = MODEL_COLORS[model]

    # Histogram
    axes[0].hist(flat_data['Reference'], bins=bins, range=(lo, hi),
                 density=True, alpha=0.45, color='gray', label='Reference')
    axes[0].hist(flat_data[model], bins=bins, range=(lo, hi),
                 density=True, alpha=0.55, color=color, label=model)
    axes[0].set_xlim(lo, hi)
    axes[0].set_title(f'Marginal distribution — {model}')
    axes[0].set_xlabel('Value (log-return)'); axes[0].set_ylabel('Density')
    axes[0].legend()

    # QQ plot
    q_ref = np.quantile(flat_data['Reference'], probs)
    q_mod = np.quantile(flat_data[model], probs)
    axes[1].scatter(q_ref, q_mod, s=3, alpha=0.6, color=color, label=model)
    axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1.0, label='y = x')
    axes[1].set_xlim(lo, hi); axes[1].set_ylim(lo, hi)
    axes[1].set_title(f'QQ plot vs Reference — {model}')
    axes[1].set_xlabel('Reference quantiles'); axes[1].set_ylabel('Model quantiles')
    axes[1].legend(markerscale=3)

    # ECDF
    ref_vals = flat_data['Reference']
    mod_vals = flat_data[model]
    s_ref = np.sort(ref_vals[(ref_vals >= lo) & (ref_vals <= hi)])
    s_mod = np.sort(mod_vals[(mod_vals >= lo) & (mod_vals <= hi)])
    axes[2].plot(s_ref, np.linspace(0, 1, len(s_ref)), color='gray', lw=1.2, label='Reference')
    axes[2].plot(s_mod, np.linspace(0, 1, len(s_mod)), color=color, lw=1.2, label=model)
    axes[2].set_xlim(lo, hi)
    axes[2].set_title(f'ECDF comparison — {model}')
    axes[2].set_xlabel('Value (log-return)'); axes[2].set_ylabel('CDF')
    axes[2].legend()

    plt.suptitle(f'Marginal distribution: {model} vs Reference', y=1.02)
    plt.tight_layout()
    save_fig(fig, f'marginal_distribution_{model.lower()}_{appendix}')
    display(fig); plt.close(fig)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18120\1301842910.py:42: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18120\4123243370.py:63: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(path, dpi=150, bbox_inches='tight')


Saved: ..\Master-Thesis\images\generated\garch_merton_comparison\marginal_distribution_garch_garch_merton_replication.png


<Figure size 1600x400 with 3 Axes>

Saved: ..\Master-Thesis\images\generated\garch_merton_comparison\marginal_distribution_merton_garch_merton_replication.png


<Figure size 1600x400 with 3 Axes>

In [33]:
lo, hi = np.quantile(flat_data['Reference'], [0.02, 0.98])
bins   = np.linspace(lo, hi, 80)
probs  = np.linspace(0.02, 0.98, 2_000)

for model in ['GARCH', 'Merton']:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    color = MODEL_COLORS[model]

    # Histogram
    axes[0].hist(flat_data['Reference'], bins=bins, range=(lo, hi),
                 density=True, alpha=0.45, color='gray', label='Reference')
    axes[0].hist(flat_data[model], bins=bins, range=(lo, hi),
                 density=True, alpha=0.55, color=color, label=model)
    axes[0].set_xlim(lo, hi)
    axes[0].set_title(f'Marginal distribution — {model}')
    axes[0].set_xlabel('Value (log-return)'); axes[0].set_ylabel('Density')
    axes[0].legend()

    # QQ plot
    q_ref = np.quantile(flat_data['Reference'], probs)
    q_mod = np.quantile(flat_data[model], probs)
    axes[1].scatter(q_ref, q_mod, s=3, alpha=0.6, color=color, label=model)
    axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1.0, label='y = x')
    axes[1].set_xlim(lo, hi); axes[1].set_ylim(lo, hi)
    axes[1].set_title(f'QQ plot vs Reference — {model}')
    axes[1].set_xlabel('Reference quantiles'); axes[1].set_ylabel('Model quantiles')
    axes[1].legend(markerscale=3)

    # ECDF
    ref_vals = flat_data['Reference']
    mod_vals = flat_data[model]
    s_ref = np.sort(ref_vals[(ref_vals >= lo) & (ref_vals <= hi)])
    s_mod = np.sort(mod_vals[(mod_vals >= lo) & (mod_vals <= hi)])
    axes[2].plot(s_ref, np.linspace(0, 1, len(s_ref)), color='gray', lw=1.2, label='Reference')
    axes[2].plot(s_mod, np.linspace(0, 1, len(s_mod)), color=color, lw=1.2, label=model)
    axes[2].set_xlim(lo, hi)
    axes[2].set_title(f'ECDF comparison — {model}')
    axes[2].set_xlabel('Value (log-return)'); axes[2].set_ylabel('CDF')
    axes[2].legend()

    plt.suptitle(f'Marginal distribution: {model} vs Reference', y=1.02)
    plt.tight_layout()
    save_fig(fig, f'THINNER_marginal_distribution_{model.lower()}_{appendix}')
    display(fig); plt.close(fig)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18120\2176234157.py:42: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18120\4123243370.py:63: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(path, dpi=150, bbox_inches='tight')


Saved: ..\Master-Thesis\images\generated\garch_merton_comparison\THINNER_marginal_distribution_garch_garch_merton_replication.png


<Figure size 1600x400 with 3 Axes>

Saved: ..\Master-Thesis\images\generated\garch_merton_comparison\THINNER_marginal_distribution_merton_garch_merton_replication.png


<Figure size 1600x400 with 3 Axes>

---
## Summary

In [31]:
print('Output tables in:', Path(OUTPUT_TABLE_DIR).resolve())
for f in sorted(Path(OUTPUT_TABLE_DIR).glob('*.csv')):
    print(f'  {f.name}')

print()
print('=' * 70)
print('  AGGREGATE STATISTICS (key columns)')
print('=' * 70)
display(df_agg[['mean', 'std', 'skewness', 'excess_kurtosis', 'q01', 'q99']].round(6))

print()
print('=' * 70)
print('  AGGREGATE STATISTICS ON INCREMENTS (key columns)')
print('=' * 70)
display(df_inc[['mean', 'std', 'skewness', 'excess_kurtosis', 'q01', 'q99']].round(6))

print()
print('=' * 70)
print('  DISTRIBUTIONAL DISTANCES vs REFERENCE')
print('=' * 70)
display(df_ks)

print()
print('=' * 70)
print('  DISTRIBUTIONAL DISTANCES ON INCREMENTS vs REFERENCE')
print('=' * 70)
display(df_ks_inc)

Output tables in: C:\Users\Lenovo\Documents\SCUOLA\UNI\MASTER\2ANNO\THESIS\Master-Thesis\tables\generated\garch_merton_comparison
  aggregate_statistics_garch_merton_replication.csv
  distributional_distances_garch_merton_replication.csv
  increment_distributional_distances_garch_merton_replication.csv
  increment_statistics_garch_merton_replication.csv

  AGGREGATE STATISTICS (key columns)


,mean,std,skewness,excess_kurtosis,q01,q99
model,,,,,,
Reference,0.000482,0.023162,-0.577267,38.502805,-0.062624,0.064912
GARCH,0.000942,0.022990,-7.563495,11021.695990,-0.057664,0.059662
Merton,0.000462,0.036449,1.210231,3736.486894,-0.058440,0.059844



  AGGREGATE STATISTICS ON INCREMENTS (key columns)


,mean,std,skewness,excess_kurtosis,q01,q99
model,,,,,,
Reference,-0.000001,0.032922,0.222717,29.409996,-0.089362,0.090966
GARCH,0.000011,0.032437,-1.751990,1500.627045,-0.081968,0.082472
Merton,0.000043,0.050706,2.098479,1850.848739,-0.079970,0.079923



  DISTRIBUTIONAL DISTANCES vs REFERENCE


,KS_stat,KS_pvalue,Wasserstein_1
model,,,
GARCH,0.0540,0.00e+00,0.001265
Merton,0.0423,0.00e+00,0.001881



  DISTRIBUTIONAL DISTANCES ON INCREMENTS vs REFERENCE


,KS_stat,KS_pvalue,Wasserstein_1
model,,,
GARCH,0.0268,0.00e+00,0.001275
Merton,0.0270,0.00e+00,0.002125


In [36]:
def fit_powerlaw_fast(returns, max_sample=300_000, seed=42):
    x = np.abs(np.asarray(returns, dtype=float))
    x = x[np.isfinite(x) & (x > 0)]
    if len(x) > max_sample:
        x = np.random.default_rng(seed).choice(x, size=max_sample, replace=False)
    x = np.sort(x)
    n = len(x)
    log_x      = np.log(x)
    rev_cumlog = np.cumsum(log_x[::-1])[::-1]
    n_tail_arr = np.arange(n, 0, -1, dtype=np.float64)
    sum_log_r  = rev_cumlog - n_tail_arr * log_x
    valid      = sum_log_r > 1e-12
    safe_denom = np.where(valid, sum_log_r, 1.0)
    alpha_arr  = np.where(valid, 1.0 + n_tail_arr / safe_denom, np.inf)
    ks_arr     = np.full(n, np.inf)
    for i in range(n - 1):
        if not valid[i]:
            continue
        m         = n - i
        a         = alpha_arr[i]
        cdf_th    = 1.0 - (x[i] / x[i:]) ** (a - 1.0)
        cdf_em    = np.arange(1, m + 1, dtype=np.float64) / m
        ks_arr[i] = np.max(np.abs(cdf_em - cdf_th))
    best = int(np.argmin(ks_arr))
    return {
        'alpha'    : float(alpha_arr[best]),
        'alpha_mle': float(alpha_arr[best]),
        'xmin'     : float(x[best]),
        'ks'       : float(ks_arr[best]),
        'n_total'  : len(x),
        'n_tail'   : n - best,
    }

pl_rows = []
for model in MODEL_ORDER:
    res = fit_powerlaw_fast(flat_data[model])
    res['model'] = model
    pl_rows.append(res)
    print(f'{model:10s}: alpha={res["alpha"]:.4f}  '
          f'xmin={res["xmin"]:.6f}  '
          f'KS={res["ks"]:.4f}  '
          f'n_tail={res["n_tail"]:,}')

df_pl = pd.DataFrame(pl_rows).set_index('model')
display(df_pl.round(4))
df_pl.to_csv(Path(OUTPUT_TABLE_DIR) / f'powerlaw_estimates_{appendix}.csv')


Reference : alpha=4.5315  xmin=0.116972  KS=0.0138  n_tail=901
GARCH     : alpha=4.1884  xmin=0.072512  KS=0.0103  n_tail=3,266
Merton    : alpha=3.3439  xmin=0.033730  KS=0.0284  n_tail=23,508


,alpha,alpha_mle,xmin,ks,n_total,n_tail
model,,,,,,
Reference,4.5315,4.5315,0.1170,0.0138,300000,901
GARCH,4.1884,4.1884,0.0725,0.0103,300000,3266
Merton,3.3439,3.3439,0.0337,0.0284,300000,23508
